In [1]:
all_programs_graphs = {}

# Furnace Robot

## Gripper V

In [2]:

import networkx as nx

program_graph = nx.DiGraph()
all_programs_graphs["gpss/furnace_robot"] = program_graph

gripper_v_position = "gpss/furnace_robot/gripper_v"
program_graph.add_node(
    gripper_v_position,
    type=("crucible", "vial", "cap", "cap_sieved", "hole_plug")
)

# Add nodes based on gripper_v (home_dac, home_consum_rack)
# consumable rack
for level in range(1, 8):
    for consum_type in ["cap", "cap_sieved", "crucible", "vial"]:
        for j in range(1, 6):
            consum_type_in_program = {
                "cap": "cap_A",
                "cap_sieved": "cap_B",
                "crucible": "crucible",
                "vial": "vial",
            }
            position = f"gpss/consumable_rack/level_{level}/row_{j}/{consum_type}"
            program_graph.add_node(position, type=(consum_type,))
            # auto_program/pick_consumable/pick_level_{level}_row_{row}_{consum}.auto.urp
            program_graph.add_edge(position, gripper_v_position, programs=[f"auto_program/pick_consumable/pick_level_{level}_row_{j}_{consum_type_in_program[consum_type]}.auto.urp"])
            # auto_program/place_consumable/place_level_{level}_row_{row}_{consum}.auto.urp
            program_graph.add_edge(gripper_v_position, position, programs=[f"auto_program/place_consumable/place_level_{level}_row_{j}_{consum_type_in_program[consum_type]}.auto.urp"])

# transfer_rack
for consum_type in ["cap", "cap_sieved", "crucible", "vial"]:
    position = f"gpss/transfer_rail/left/{consum_type}"
    program_graph.add_node(position, type=(consum_type,))
    # translate the consum_type to the consum_type_trans
    consum_type_in_program = {
        "cap": "cap_A",
        "cap_sieved": "cap_B",
        "crucible": "crucible",
        "vial": "vial",
    }
    # auto_program/pick_trans_rack/pick_trans_rack_{consum}.auto.urp
    program_graph.add_edge(position, gripper_v_position, programs=[f"auto_program/pick_trans_rack/pick_trans_rack_{consum_type_in_program[consum_type]}.auto.urp"])
    # auto_program/place_trans_rack/place_trans_rack_{consum}.auto.urp
    program_graph.add_edge(gripper_v_position, position, programs=[f"auto_program/place_trans_rack/place_trans_rack_{consum_type_in_program[consum_type]}.auto.urp"])

# crucible on furnace rack loading
for i in range(1, 9):
    position = f"gpss/furnace_rack_loading/crucible/{i}"
    program_graph.add_node(position, type=("crucible",))
    # auto_program/place_crucible_furnace_rack/place_crucible_furnace_rack_{pos}.auto.urp
    program_graph.add_edge(gripper_v_position, position, programs=[f"auto_program/place_crucible_furnace_rack/place_crucible_furnace_rack_{i}.auto.urp"])
    # auto_program/pick_crucible_furnace_rack/pick_crucible_furnace_rack_{pos}.auto.urp
    program_graph.add_edge(position, gripper_v_position, programs=[f"auto_program/pick_crucible_furnace_rack/pick_crucible_furnace_rack_{i}.auto.urp"])

# cru near ball_dispenser
position = "gpss/cru_holder_near_ball_dispenser/crucible"
program_graph.add_node(position, type=("crucible",))
# auto_program/pick_crucible_near_bdis.auto.urp
program_graph.add_edge(position, gripper_v_position, programs=["auto_program/pick_crucible_near_bdis.auto.urp"])
# auto_program/place_crucible_near_bdis.auto.urp
program_graph.add_edge(gripper_v_position, position, programs=["auto_program/place_crucible_near_bdis.auto.urp"])

# crucible on capper
position = "gpss/crucible_capper/crucible"
program_graph.add_node(position, type=("crucible",))
# auto_program/pick_cru_capper.auto.urp
program_graph.add_edge(position, gripper_v_position, programs=["auto_program/pick_cru_capper.auto.urp"])
# auto_program/place_cru_capper.auto.urp
program_graph.add_edge(gripper_v_position, position, programs=["auto_program/place_cru_capper.auto.urp"])

# dac crucible
position = "gpss/dac/crucible"
program_graph.add_node(position, type=("crucible",))
# auto_program/pick_cru_dac.auto.urp
program_graph.add_edge(position, gripper_v_position, programs=["auto_program/pick_cru_dac.auto.urp"])
# auto_program/place_cru_dac.auto.urp
program_graph.add_edge(gripper_v_position, position, programs=["auto_program/place_cru_dac.auto.urp"])

# vial on shaker vial
position = "gpss/shaker/vial"
program_graph.add_node(position, type=("vial",))
program_graph.add_edge(position, gripper_v_position, programs=["auto_program/pick_shaker_vial.auto.urp"])
program_graph.add_edge(gripper_v_position, position, programs=["auto_program/place_shaker_vial.auto.urp"])

# hole plug holders
for pos in ["A", "B", "C", "D"]:
    position = f"gpss/hole_plug_holder/hole_plug/{pos}"
    program_graph.add_node(position, type=("hole_plug",))
    # auto_program/pick_hole_plug/pick_hole_plug_{pos}.auto.urp
    program_graph.add_edge(position, gripper_v_position, programs=[f"auto_program/pick_hole_plug/pick_hole_plug_{pos}.auto.urp"])
    # auto_program/place_hole_plug/place_hole_plug_{pos}.auto.urp
    program_graph.add_edge(gripper_v_position, position, programs=[f"auto_program/place_hole_plug/place_hole_plug_{pos}.auto.urp"])

## Gripper HF

In [3]:
gripper_hf_position = "gpss/furnace_robot/gripper_hf"
program_graph.add_node(gripper_hf_position, type=("furnace_rack",))

# furnace rack loading
for pos in ["loading", "A", "B", "in_fur_A", "in_fur_B"]:
    if pos == "loading":
        position = "gpss/furnace_rack_loading/furnace_rack"
    elif pos in {"A", "B"}:
        position = f"gpss/furnace_{pos}/cooling_area/furnace_rack"
    else:
        letter = pos[-1]
        position = f"gpss/furnace_{letter}/furnace_rack"
    program_graph.add_node(position, type=("furnace_rack",))
    # auto_program/pick_furnace_rack_loading/pick_furnace_rack_loading_{pos}.auto.urp
    if pos in {"A", "B"}:
        pick_program = f"auto_program/pick_furnace_rack_f_side/pick_furnace_rack_{pos}.auto.urp"
        place_program = f"auto_program/place_furnace_rack_f_side/place_furnace_rack_{pos}.auto.urp"
    else:
        pick_program = f"auto_program/pick_furnace_rack_{pos}.auto.urp"
        place_program = f"auto_program/place_furnace_rack_{pos}.auto.urp"
    program_graph.add_edge(position, gripper_hf_position, programs=["auto_program/pick_furnace_handle.auto.urp", pick_program])
    # auto_program/place_furnace_rack_loading/place_furnace_rack_loading_{pos}.auto.urp
    program_graph.add_edge(gripper_hf_position, position, programs=[place_program, "auto_program/place_furnace_handle.auto.urp"])

## Gripper HDac

In [4]:
gripper_hdac_position = "gpss/furnace_robot/gripper_hdac"
program_graph.add_node(gripper_hdac_position, type=("crucible", "dac_lid"))

# DAC lid
position_1 = "gpss/dac/dac_lid"
position_2 = "gpss/dac_lid_holder/dac_lid"
program_graph.add_node(position_1, type=("dac_lid",))
program_graph.add_node(position_2, type=("dac_lid",))
program_graph.add_edge(position_1, position_2, programs=["auto_program/take_lid_on_dac.auto.urp"])
program_graph.add_edge(position_2, position_1, programs=["auto_program/put_lid_on_dac.auto.urp"])

# ball dispenser
position = "gpss/ball_dispenser/crucible"
program_graph.add_node(position, type=("crucible",))
program_graph.add_edge(position, "gpss/furnace_robot/gripper_hdac", programs=["auto_program/after_ball_dispensing.auto.urp"])
program_graph.add_edge("gpss/furnace_robot/gripper_hdac", position, programs=["auto_program/before_ball_dispensing.auto.urp"])

# shaker crucible
position = "gpss/shaker/crucible"
program_graph.add_node(position, type=("crucible",))
program_graph.add_edge(position, "gpss/furnace_robot/gripper_hdac", programs=["auto_program/unload_cru_shaker.auto.urp"])
program_graph.add_edge("gpss/furnace_robot/gripper_hdac", position, programs=["auto_program/load_cru_shaker.auto.urp"])

# crucible dumping
position = "gpss/shaker/dumping/crucible"
program_graph.add_node(position, type=("crucible",))
program_graph.add_edge(position, "gpss/furnace_robot/gripper_hdac", programs=["auto_program/after_dumping.auto.urp"])
program_graph.add_edge("gpss/furnace_robot/gripper_hdac", position, programs=["auto_program/before_dumping.auto.urp"])

# crucible near ball dispenser
position = "gpss/cru_holder_near_ball_dispenser/crucible"
program_graph.add_node(position, type=("crucible",))
program_graph.add_edge(position, "gpss/furnace_robot/gripper_hdac", programs=["auto_program/pick_crucible_by_side.auto.urp"])
program_graph.add_edge("gpss/furnace_robot/gripper_hdac", position, programs=["auto_program/place_crucible_by_side.auto.urp"])


## Visualize the graph

In [5]:
# # Visualize the graph
# import networkx as nx
# import matplotlib.pyplot as plt


# def visualize_graph(program_graph):
#     # Create a new graph with merged nodes
#     merged_graph = nx.DiGraph()

#     # Helper function to get base path
#     def get_base_path(node):
#         parts = node.split('/')
#         return '/'.join(parts[:2]) if len(parts) > 1 else node

#     # Merge nodes and edges
#     for node in program_graph.nodes():
#         base = get_base_path(node)
#         if not merged_graph.has_node(base):
#             merged_graph.add_node(base)

#     for u, v, data in program_graph.edges(data=True):
#         u_base = get_base_path(u)
#         v_base = get_base_path(v)
#         if not merged_graph.has_edge(u_base, v_base):
#             merged_graph.add_edge(u_base, v_base)

#     plt.figure(figsize=(15, 10))
#     pos = nx.spring_layout(merged_graph, k=2, iterations=50)
#     nx.draw(merged_graph, pos, with_labels=True, node_color='lightblue',
#             node_size=2000, font_size=8, font_weight='bold')

#     plt.title("Robot Arm Program Graph (Merged)")
#     plt.axis('off')
#     plt.show()

# visualize_graph(program_graph)


## check if the program is valid

In [6]:
from pathlib import Path

auto_program_path = Path("/Users/yuxing/Downloads/gpss_robot_backup_041025/programs_gpss-2_041025/auto_program")
gpss_2_programs = list(auto_program_path.glob("**/*.auto.urp"))

exclude_program_pattern = [
    "auto_program/close_consumable_rack",
    "auto_program/open_consumable_rack",
    "auto_program/capping.auto.urp",
    "auto_program/decapping.auto.urp",
]

gpss_2_programs = [p.relative_to(auto_program_path.parent).as_posix() for p in gpss_2_programs if "touch" not in p.name and not any(pattern in p.as_posix() for pattern in exclude_program_pattern)]


# extract all the program from the graph
program_graph_programs = [data["programs"] for _, _, data in program_graph.edges(data=True)]
program_graph_programs = [item.split(":")[-1] for sublist in program_graph_programs for item in sublist]

# check if the program is valid
extra_programs = set(program_graph_programs) - set(gpss_2_programs)
if len(extra_programs) > 0:
    print(f"There are {len(extra_programs)} programs that are not valid")
    print(f"The invalid programs are: {extra_programs}")
else:
    print("All programs are valid")

missing_programs = set(gpss_2_programs) - set(program_graph_programs)
if len(missing_programs) > 0:
    print(f"There are {len(missing_programs)} programs that are not in the graph")
    print(f"The missing programs are: {missing_programs}")
else:
    print("All programs are valid")




All programs are valid
There are 1 programs that are not in the graph
The missing programs are: {'auto_program/pick_cru_by_side.auto.urp'}


# Powder robot

## gripper_dosing_head

In [7]:
program_graph = nx.DiGraph()
all_programs_graphs["gpss/powder_robot"] = program_graph

gripper_dosing_head_position = "gpss/powder_robot/gripper_dosing_head"
program_graph.add_node(gripper_dosing_head_position, type=("dosing_head",))

# dosing head rack
for pos in range(1, 15):
    for slot in ["A", "B", "C", "D"]:
        position = f"gpss/dosing_head_rack/{pos}{slot}"
        program_graph.add_node(position, type=("dosing_head",))
        # auto_program/pick_dosing_head_rack/pick_dosing_head_rack_{pos}_{slot}.auto.urp
        program_graph.add_edge(position, gripper_dosing_head_position, programs=[f"auto_program/pick_dose_head_{slot}.auto.urp"])
        # auto_program/place_dosing_head_rack/place_dosing_head_rack_{pos}_{slot}.auto.urp
        program_graph.add_edge(gripper_dosing_head_position, position, programs=[f"auto_program/place_dose_head_{slot}.auto.urp"])


# dosing head balance
position = "gpss/auto_balance/dosing_head"
program_graph.add_node(position, type=("dosing_head",))
program_graph.add_edge(position, gripper_dosing_head_position, programs=["auto_program/pick_dose_head_balance.auto.urp"])
program_graph.add_edge(gripper_dosing_head_position, position, programs=["auto_program/place_dose_head_balance.auto.urp"])



## gripper

In [8]:
gripper_position = "gpss/powder_robot/gripper_v"
program_graph.add_node(gripper_position, type=("cap", "cap_sieved", "crucible", "vial", "xrd_sample_holder"))

# crucible auto balance
position = "gpss/auto_balance/crucible"
program_graph.add_node(position, type=("crucible",))
program_graph.add_edge(position, gripper_position, programs=["auto_program/pick_cru_balance.auto.urp"])
program_graph.add_edge(gripper_position, position, programs=["auto_program/place_cru_balance.auto.urp"])

# transfer rail
for consum_type in ["cap", "cap_sieved", "crucible", "vial"]:
    consum_type_in_program = {
        "cap": "cap_A",
        "cap_sieved": "cap_B",
        "crucible": "cru",
        "vial": "vial",
    }
    position = f"gpss/transfer_rail/right/{consum_type}"
    program_graph.add_node(position, type=(consum_type,))
    # auto_program/pick_trans_rail/pick_trans_rail_{consum}.auto.urp
    program_graph.add_edge(position, gripper_position, programs=[f"auto_program/pick_trans_rack_{consum_type_in_program[consum_type]}.auto.urp"])
    # auto_program/place_trans_rail/place_trans_rail_{consum}.auto.urp
    program_graph.add_edge(gripper_position, position, programs=[f"auto_program/place_trans_rack_{consum_type_in_program[consum_type]}.auto.urp"])

# vial capper
position = "gpss/vial_capper/vial"
program_graph.add_node(position, type=("vial",))
program_graph.add_edge(position, gripper_position, programs=["auto_program/pick_vial_capper.auto.urp"])
program_graph.add_edge(gripper_position, position, programs=["auto_program/place_vial_capper.auto.urp"])

# cap holder
for consum_type, pos in [("cap", "A"), ("cap_sieved", "B")]:
    position = f"gpss/cap_holder/{consum_type}"
    program_graph.add_node(position, type=(consum_type,))
    program_graph.add_edge(position, gripper_position, programs=[f"auto_program/pick_cap_{pos}.auto.urp"])
    program_graph.add_edge(gripper_position, position, programs=[f"auto_program/place_cap_{pos}.auto.urp"])

# XRD sample holder rack
for rack in ["A", "B", "C", "D"]:
    for slot in [1, 2, 3, 4]:
        position = f"gpss/xrd_sample_holder_rack/xrd_sample_holder/{rack}{slot}"
        program_graph.add_node(position, type=("xrd_sample_holder",))
        program_graph.add_edge(position, gripper_position, programs=[f"auto_program/pick_xrd_holder_rack/pick_xrd_holder_rack_{rack}{slot}.auto.urp"])
        program_graph.add_edge(gripper_position, position, programs=[f"auto_program/place_xrd_holder_rack/place_xrd_holder_rack_{rack}{slot}.auto.urp"])

# XRD dispenser XRD sample holder
position = "gpss/xrd_dispenser/xrd_sample_holder"
program_graph.add_node(position, type=("xrd_sample_holder"))
program_graph.add_edge(position, gripper_position, programs=["auto_program/pick_xrd_holder_dispenser.auto.urp"])
program_graph.add_edge(gripper_position, position, programs=["auto_program/place_xrd_holder_dispenser.auto.urp"])

# XRD dispenser vial
position = "gpss/xrd_dispenser/vial"
program_graph.add_node(position, type=("vial",))
program_graph.add_edge(position, gripper_position, programs=["auto_program/pick_xrd_disp_vial.auto.urp"])
program_graph.add_edge(gripper_position, position, programs=["auto_program/place_xrd_disp_vial.auto.urp"])

In [9]:
from pathlib import Path

auto_program_path = Path("/Users/yuxing/Downloads/gpss_robot_backup_041025/programs_gpss-1_041025/auto_program")
gpss_1_programs = list(auto_program_path.glob("**/*.auto.urp"))

exclude_program_pattern = [
    "capping.auto.urp",
    "decapping.auto.urp",
    "dispose.auto.urp",
]

gpss_1_programs = [p.relative_to(auto_program_path.parent).as_posix() for p in gpss_1_programs if not any(pattern in p.as_posix() for pattern in exclude_program_pattern)]

# extract all the program from the graph
program_graph_programs = [data["programs"]for _, _, data in program_graph.edges(data=True)]
program_graph_programs = [item.split(": ")[-1] for sublist in program_graph_programs for item in sublist]

# check if the program is valid
extra_programs = set(program_graph_programs) - set(gpss_1_programs)
if len(extra_programs) > 0:
    print(f"There are {len(extra_programs)} programs that are not valid")
    print(f"The invalid programs are: {extra_programs}")
else:
    print("All programs are valid")

missing_programs = set(gpss_1_programs) - set(program_graph_programs)
if len(missing_programs) > 0:
    print(f"There are {len(missing_programs)} programs that are not in the graph")
    print(f"The missing programs are: {missing_programs}")
else:
    print("All programs are valid")




All programs are valid
There are 2 programs that are not in the graph
The missing programs are: {'auto_program/place_crucible_dispenser.auto.urp', 'auto_program/pick_crucible_dispenser.auto.urp'}


# Merge all graphs

In [10]:
import pickle

for robot, graph in all_programs_graphs.items():
    # add the name of the robot as a edge attribute
    for u, v, data in graph.edges(data=True):
        data["robot"] = robot

composed_graph = nx.compose_all(all_programs_graphs.values())
composed_graph

# validate that for each consumble, there is only one connected graph
for consum_type in ["cap", "cap_sieved", "crucible", "vial", "xrd_sample_holder", "dac_lid", "furnace_rack"]:
    sub_graph = nx.subgraph_view(composed_graph, filter_node=lambda n: consum_type in nx.get_node_attributes(composed_graph, "type", default=tuple())[n])
    # translate the sub_graph into an undirected graph
    sub_graph = sub_graph.to_undirected()
    # check if the sub_graph is connected
    if not nx.is_connected(sub_graph):
        print(f"The sub_graph for {consum_type} is not connected")

# save the graph as a pickle
pickle.dump(composed_graph, open("moving_graph.pkl", "wb"))

The sub_graph for cap is not connected
The sub_graph for cap_sieved is not connected
The sub_graph for crucible is not connected
The sub_graph for vial is not connected
The sub_graph for dac_lid is not connected


In [17]:
composed_graph = pickle.load(open("moving_graph.pkl", "rb"))

# find a path from the start to the end
start = "gpss/furnace_rack_loading/crucible/3"
end = "gpss/furnace_robot/gripper_v"

# filter by consumble type, here we filter by crucible
sub_graph = nx.subgraph_view(composed_graph, filter_node=lambda n: "crucible" in nx.get_node_attributes(composed_graph, "type", default=tuple())[n])

path = nx.shortest_path(sub_graph, start, end)
# translate the path into a list of (robot, program_list)
program_list = []

for i in range(len(path) - 1):
    program = sub_graph.get_edge_data(path[i], path[i + 1])
    if program_list and program_list[-1]["robot"] == program["robot"]:
        program_list[-1]["program_list"].append(
            {
                "start": path[i],
                "end": path[i + 1],
                "programs": program["programs"],
            }
        )
    else:
        program_list.append(
            {
                "robot": program["robot"],
                "program_list": [
                    {
                        "start": path[i],
                        "end": path[i + 1],
                        "programs": program["programs"],
                    }
                ],
            }
        )

request = {None: {}}
for position in path:
    position_processed = position
    parent_device = None
    request[parent_device][
        position_processed
    ] = 1  # need one position per parent.

for program in program_list:
    robot = program["robot"]
    if robot not in request:
        request[robot] = {}


program_list

[{'robot': 'gpss/furnace_robot',
  'program_list': [{'start': 'gpss/furnace_rack_loading/crucible/3',
    'end': 'gpss/furnace_robot/gripper_v',
    'programs': ['auto_program/pick_crucible_furnace_rack/pick_crucible_furnace_rack_3.auto.urp']}]}]

In [12]:
program_list

[{'robot': 'gpss/furnace_robot',
  'program_list': [{'start': 'gpss/furnace_rack_loading/crucible/3',
    'end': 'gpss/furnace_robot/gripper_v',
    'programs': ['auto_program/pick_crucible_furnace_rack/pick_crucible_furnace_rack_3.auto.urp']}]}]

In [13]:
composed_graph.nodes["gpss/auto_balance/crucible"]

{'type': ('crucible',)}

In [14]:
for node in composed_graph.nodes:
    if node.startswith("gpss/furnace_A"):
        print(node)


gpss/furnace_A/cooling_area/furnace_rack
gpss/furnace_A/furnace_rack


In [15]:
"gpss/furnace_robot/gripper_v" in sub_graph

True

# Test